In [1]:
!nvidia-smi

Sun Oct  6 09:26:29 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.183.06             Driver Version: 535.183.06   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off | 00000000:B7:00.0 Off |                    0 |
| N/A   42C    P0              59W / 400W |      0MiB / 40960MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [ ]:
!pip install transformers -U


In [1]:
import sys
import os
os.getcwd()

sys.path.append('/home/huuthanhvy.nguyen001/LLMP/')

import torch
from transformers import TrainingArguments, Trainer, TrainerCallback, MllamaForConditionalGeneration
from peft import LoraConfig, get_peft_model
import matplotlib.pyplot as plt

# Load model and processor as before
from transformers import MllamaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# Model ID
model_id = "raminguyen/rami-llama-finetuned1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
)
processor = AutoProcessor.from_pretrained(model_id)

model.tie_weights()

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
/home/huuthanhvy.nguyen001/anaconda3/envs/pytorch/lib/python3.11/site-packages/transformers/quantizers/auto.py:182: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [2]:
import torch
print(torch.__version__)

2.4.1


In [2]:
from PIL import Image
import torch
import torchvision.transforms as transforms
import numpy as np

# Prepare your input data
input_question = "Describe the image content."
input_image_path = "0cf01149-0188-41bc-b0bf-2293d6f5275d.jpg"

# Prepare the text input
text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n<|image|>{input_question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

# Load and preprocess the image
image = Image.open(input_image_path).convert("RGB")
print(f"Original image size: {image.size}")

# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Preprocess the image
preprocess = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
])

image = preprocess(image)
print(f"Image shape after preprocessing: {image.shape}")

# Convert to numpy array and adjust dimensions
image_np = image.numpy()
image_np = np.transpose(image_np, (1, 2, 0))  # Change from (C, H, W) to (H, W, C)
image_np = np.expand_dims(image_np, axis=0)  # Add batch dimension
print(f"Image shape after numpy conversion: {image_np.shape}")

You shouldn't move a model that is dispatched using accelerate hooks.


Original image size: (100, 100)
Image shape after preprocessing: torch.Size([3, 448, 448])
Image shape after numpy conversion: (1, 448, 448, 3)


In [5]:
from PIL import Image
import torch
import torchvision.transforms as transforms
import numpy as np

# Prepare your input data
input_question = "Describe the image content."
input_image_path = "0cf01149-0188-41bc-b0bf-2293d6f5275d.jpg"

# Prepare the text input
text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n<|image|>{input_question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

# Load and preprocess the image
image = Image.open(input_image_path).convert("RGB")
print(f"Original image size: {image.size}")

inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True)

inputs

Original image size: (100, 100)


{'input_ids': tensor([[128000, 128000, 128006,    882, 128007,    271, 128256,  75885,    279,
           2217,   2262,     13, 128009, 128006,  78191, 128007,    271]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'pixel_values': tensor([[[[[[-1.7923, -1.7923, -1.7923,  ..., -1.7923, -1.7923, -1.7923],
            [-1.7923, -1.7923, -1.7923,  ..., -1.7923, -1.7923, -1.7923],
            [-1.7923, -1.7923, -1.7923,  ..., -1.7923, -1.7923, -1.7923],
            ...,
            [-1.7631, -1.7631, -1.7631,  ..., -1.5733, -1.7923, -1.7923],
            [-1.7631, -1.7631, -1.7631,  ..., -1.5733, -1.7923, -1.7923],
            [-1.7631, -1.7631, -1.7631,  ..., -1.5733, -1.7923, -1.7923]],

           [[-1.7521, -1.7521, -1.7521,  ..., -1.7521, -1.7521, -1.7521],
            [-1.7521, -1.7521, -1.7521,  ..., -1.7521, -1.7521, -1.7521],
            [-1.7521, -1.7521, -1.7521,  ..., -1.7521, -1.7521, -1.7521],
            ...,
            [-1.7221, -1.7221,

In [6]:
output = model.generate(**inputs, max_new_tokens=20)


/home/huuthanhvy.nguyen001/anaconda3/envs/pytorch/lib/python3.11/site-packages/transformers/generation/utils.py:2026: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


RuntimeError: mat1 and mat2 shapes cannot be multiplied (17x4096 and 1024x4096)

In [ ]:
!pip uninstall bitsandbytes -y

!pip install bitsandbytes -y

In [28]:
print(model.config)

MllamaConfig {
  "_name_or_path": "raminguyen/rami-llama-finetuned1",
  "architectures": [
    "MllamaForConditionalGeneration"
  ],
  "image_token_index": 128256,
  "model_type": "mllama",
  "quantization_config": {
    "_load_in_4bit": true,
    "_load_in_8bit": false,
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_quant_storage": "uint8",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "llm_int8_enable_fp32_cpu_offload": false,
    "llm_int8_has_fp16_weight": false,
    "llm_int8_skip_modules": null,
    "llm_int8_threshold": 6.0,
    "load_in_4bit": true,
    "load_in_8bit": false,
    "quant_method": "bitsandbytes"
  },
  "text_config": {
    "_name_or_path": "",
    "add_cross_attention": false,
    "architectures": null,
    "bad_words_ids": null,
    "begin_suppress_tokens": null,
    "bos_token_id": 128000,
    "chunk_size_feed_forward": 0,
    "cross_attention_hidden_size": null,
    "cross_attention_layers": [
      3,
      8,
      13,

In [19]:
!nvidia-smi

Sun Oct  6 09:51:57 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.183.06             Driver Version: 535.183.06   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off | 00000000:B7:00.0 Off |                    0 |
| N/A   43C    P0              65W / 400W |  40331MiB / 40960MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
import requests
import torch
from PIL import Image
from transformers import MllamaForConditionalGeneration, AutoProcessor

model_id = "meta-llama/Llama-3.2-11B-Vision"

model = MllamaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/0052a70beed5bf71b92610a43a52df6d286cd5f3/diffusers/rabbit.jpg"
image = Image.open(requests.get(url, stream=True).raw)

prompt = "<|image|><|begin_of_text|>If I had to write a haiku for this one"
inputs = processor(image, prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=30)
print(processor.decode(output[0]))

for key, value in inputs.items():
    print(f"{key}: shape {value.shape}, dtype {value.dtype}")

print(model.config)